In [1]:
import pandas as pd
import numpy as np
import json
import os

print("ATT&CK Mapping Notebook")
print("="*50)

ATT&CK Mapping Notebook


In [2]:
# Map each attack tactic to common techniques (from MITRE ATT&CK)
attack_mapping = {
    "defense_evasion": {
        "tactic_id": "TA0005",
        "tactic_name": "Defense Evasion",
        "description": "Adversary tries to avoid being detected",
        "common_techniques": {
            "T1055": "Process Injection",
            "T1027": "Obfuscated Files or Information",
            "T1070": "Indicator Removal on Host",
            "T1112": "Modify Registry",
            "T1218": "Signed Binary Proxy Execution",
            "T1562": "Impair Defenses"
        }
    },
    "lateral_movement": {
        "tactic_id": "TA0008",
        "tactic_name": "Lateral Movement",
        "description": "Adversary tries to move through your environment",
        "common_techniques": {
            "T1021": "Remote Services",
            "T1021.001": "Remote Desktop Protocol",
            "T1021.002": "SMB/Windows Admin Shares",
            "T1021.006": "Windows Remote Management (WinRM)",
            "T1570": "Lateral Tool Transfer",
            "T1550": "Use Alternate Authentication Material"
        }
    },
    "persistence": {
        "tactic_id": "TA0003",
        "tactic_name": "Persistence",
        "description": "Adversary tries to maintain their foothold",
        "common_techniques": {
            "T1547": "Boot or Logon Autostart Execution",
            "T1547.001": "Registry Run Keys / Startup Folder",
            "T1053": "Scheduled Task/Job",
            "T1053.005": "Scheduled Task",
            "T1136": "Create Account",
            "T1543": "Create or Modify System Process",
            "T1505": "Server Software Component"
        }
    }
}

print("ATT&CK Mapping Loaded ✅")
print(f"\nTactics covered: {len(attack_mapping)}")
total_techniques = sum(len(v['common_techniques']) for v in attack_mapping.values())
print(f"Total techniques: {total_techniques}")

ATT&CK Mapping Loaded ✅

Tactics covered: 3
Total techniques: 19


In [3]:
# Map Windows EventIDs to likely ATT&CK techniques
eventid_to_technique = {
    # Process events
    1: ["T1055", "T1218"],          # Sysmon process create
    4688: ["T1055", "T1218"],       # Security log process create
    
    # Authentication events  
    4624: ["T1021", "T1550"],       # Successful logon (lateral movement)
    4625: ["T1110"],                 # Failed logon
    4634: ["T1021"],                 # Logoff
    4648: ["T1550"],                 # Logon using explicit credentials
    
    # Registry events
    13: ["T1112", "T1547.001"],     # Sysmon registry modification
    14: ["T1112"],                   # Sysmon registry rename
    
    # File events
    11: ["T1547", "T1027"],         # Sysmon file create
    23: ["T1070"],                   # Sysmon file delete
    
    # Service/Task events
    4697: ["T1543"],                 # New service installed
    4698: ["T1053.005"],            # Scheduled task created
    4699: ["T1053.005"],            # Scheduled task deleted
    4700: ["T1053.005"],            # Scheduled task enabled
    
    # Network events
    3: ["T1021"],                    # Sysmon network connection
    5156: ["T1021"],                # Windows filtering platform
    
    # Account events
    4720: ["T1136"],                # User account created
    4732: ["T1136"],                # Member added to security group
    
    # PowerShell
    4103: ["T1059.001"],            # PowerShell module logging
    4104: ["T1059.001"],            # PowerShell script block
}

print(f"EventID mappings loaded: {len(eventid_to_technique)} EventIDs")

EventID mappings loaded: 20 EventIDs


In [4]:
# Load your rich features dataset
df = pd.read_csv(r"C:\Research\EndpointXAI\data\mordor_rich_features.csv")

# Add ATT&CK enrichment columns
def get_techniques(row):
    """Map a row to specific ATT&CK techniques based on EventID + tactic"""
    tactic = row['attack_label']
    event_id = row['EventID']
    
    # Get techniques from EventID mapping
    eventid_techs = eventid_to_technique.get(event_id, [])
    
    # Filter to only techniques that belong to this tactic
    if tactic in attack_mapping:
        tactic_techs = list(attack_mapping[tactic]['common_techniques'].keys())
        # Match EventID-suggested techniques with tactic-allowed techniques
        matched = [t for t in eventid_techs if any(t.startswith(tt.split('.')[0]) for tt in tactic_techs)]
        return ", ".join(matched) if matched else "Generic-" + attack_mapping[tactic]['tactic_id']
    return "Unknown"

def get_tactic_id(row):
    tactic = row['attack_label']
    return attack_mapping.get(tactic, {}).get('tactic_id', 'Unknown')

def get_tactic_name(row):
    tactic = row['attack_label']
    return attack_mapping.get(tactic, {}).get('tactic_name', 'Unknown')

print("Applying ATT&CK mapping... ⏳")
df['attack_tactic_id'] = df.apply(get_tactic_id, axis=1)
df['attack_tactic_name'] = df.apply(get_tactic_name, axis=1)
df['attack_techniques'] = df.apply(get_techniques, axis=1)

print("Mapping complete ✅")
print(f"\nSample of enriched data:")
print(df[['EventID', 'attack_label', 'attack_tactic_id', 'attack_techniques']].head(10))

Applying ATT&CK mapping... ⏳
Mapping complete ✅

Sample of enriched data:
   EventID     attack_label attack_tactic_id attack_techniques
0       64  defense_evasion           TA0005    Generic-TA0005
1       64  defense_evasion           TA0005    Generic-TA0005
2       64  defense_evasion           TA0005    Generic-TA0005
3       64  defense_evasion           TA0005    Generic-TA0005
4       64  defense_evasion           TA0005    Generic-TA0005
5       64  defense_evasion           TA0005    Generic-TA0005
6       64  defense_evasion           TA0005    Generic-TA0005
7       64  defense_evasion           TA0005    Generic-TA0005
8       64  defense_evasion           TA0005    Generic-TA0005
9       64  defense_evasion           TA0005    Generic-TA0005


In [5]:
# How many events were mapped to each technique?
print("Technique distribution across the dataset:\n")
technique_counts = df['attack_techniques'].value_counts()
print(technique_counts.head(20))

Technique distribution across the dataset:

attack_techniques
T1021             17917
Generic-TA0005    17390
Generic-TA0008     8076
Generic-TA0003     2046
T1547.001           189
T1547                 4
Name: count, dtype: int64


In [6]:
def explain_alert_attck(row, shap_top_features=None):
    """Generate a SOC-analyst-ready alert explanation with ATT&CK context"""
    tactic = row['attack_label']
    tactic_info = attack_mapping.get(tactic, {})
    
    explanation = []
    explanation.append("="*60)
    explanation.append(f"🚨 SECURITY ALERT")
    explanation.append("="*60)
    explanation.append(f"\n📋 Detected Tactic:")
    explanation.append(f"   {tactic_info.get('tactic_id', 'N/A')} - {tactic_info.get('tactic_name', tactic)}")
    explanation.append(f"   {tactic_info.get('description', '')}")
    
    explanation.append(f"\n🎯 Likely ATT&CK Techniques:")
    techniques = row.get('attack_techniques', 'Unknown').split(", ")
    for tech in techniques:
        if tech in tactic_info.get('common_techniques', {}):
            explanation.append(f"   {tech}: {tactic_info['common_techniques'][tech]}")
        else:
            explanation.append(f"   {tech}")
    
    explanation.append(f"\n🔍 Event Details:")
    explanation.append(f"   EventID: {row.get('EventID', 'N/A')}")
    explanation.append(f"   Channel: {row.get('Channel', 'N/A')}")
    explanation.append(f"   Source: {row.get('SourceName', 'N/A')}")
    
    if shap_top_features:
        explanation.append(f"\n🧠 AI Reasoning (Top SHAP Contributors):")
        for feat, val in shap_top_features:
            sign = "+" if val > 0 else ""
            explanation.append(f"   {feat}: {sign}{val:.2f}")
    
    explanation.append(f"\n💡 Recommended Action:")
    if tactic == "defense_evasion":
        explanation.append("   - Investigate process injection or registry modifications")
        explanation.append("   - Check for disabled security tools")
    elif tactic == "lateral_movement":
        explanation.append("   - Review authentication logs and remote sessions")
        explanation.append("   - Check for unusual network connections")
    elif tactic == "persistence":
        explanation.append("   - Audit scheduled tasks and registry run keys")
        explanation.append("   - Look for new services or accounts")
    
    explanation.append("="*60)
    return "\n".join(explanation)

# Demo on one alert
sample_alert = df.iloc[0]
print(explain_alert_attck(sample_alert))

🚨 SECURITY ALERT

📋 Detected Tactic:
   TA0005 - Defense Evasion
   Adversary tries to avoid being detected

🎯 Likely ATT&CK Techniques:
   Generic-TA0005

🔍 Event Details:
   EventID: 64
   Channel: unknown
   Source: unknown

💡 Recommended Action:
   - Investigate process injection or registry modifications
   - Check for disabled security tools


In [7]:
import joblib

# Load model and SHAP values
xgb_model = joblib.load(r"C:\Research\EndpointXAI\models\xgboost.pkl")
shap_values = np.load(r"C:\Research\EndpointXAI\xai\shap_values.npy")
X_sample = pd.read_csv(r"C:\Research\EndpointXAI\xai\shap_sample.csv")
label_encoder = joblib.load(r"C:\Research\EndpointXAI\models\label_encoder.pkl")

# Pick alert #0 from the SHAP sample
alert_idx = 0
predicted_class = xgb_model.predict(X_sample.iloc[[alert_idx]])[0]
predicted_label = label_encoder.classes_[predicted_class]

# Get top 3 SHAP contributors for this prediction
feature_names = X_sample.columns.tolist()
shap_for_alert = shap_values[predicted_class][alert_idx]
top_indices = np.argsort(np.abs(shap_for_alert))[::-1][:3]
top_features = [(feature_names[i], shap_for_alert[i]) for i in top_indices]

# Build a row dict
alert_row = X_sample.iloc[alert_idx].to_dict()
alert_row['attack_label'] = predicted_label
alert_row['attack_techniques'] = "T1055, T1218"  # example

# Generate the full explanation
print(explain_alert_attck(alert_row, shap_top_features=top_features))

🚨 SECURITY ALERT

📋 Detected Tactic:
   TA0005 - Defense Evasion
   Adversary tries to avoid being detected

🎯 Likely ATT&CK Techniques:
   T1055: Process Injection
   T1218: Signed Binary Proxy Execution

🔍 Event Details:
   EventID: 64.0
   Channel: 7.0
   Source: 7.0

🧠 AI Reasoning (Top SHAP Contributors):
   EventID: +7.89
   Task: +0.75
   AccountName: +0.51

💡 Recommended Action:
   - Investigate process injection or registry modifications
   - Check for disabled security tools


In [8]:
# Save the enriched dataset
df.to_csv(r"C:\Research\EndpointXAI\data\mordor_attck_enriched.csv", index=False)

# Save the mappings as JSON for reuse
os.makedirs(r"C:\Research\EndpointXAI\attck", exist_ok=True)

with open(r"C:\Research\EndpointXAI\attck\attack_mapping.json", "w") as f:
    json.dump(attack_mapping, f, indent=2)

with open(r"C:\Research\EndpointXAI\attck\eventid_mapping.json", "w") as f:
    json.dump(eventid_to_technique, f, indent=2)

print("All ATT&CK artifacts saved ✅")
print(f"\nFiles created:")
print(f"  - data/mordor_attck_enriched.csv")
print(f"  - attck/attack_mapping.json")
print(f"  - attck/eventid_mapping.json")

All ATT&CK artifacts saved ✅

Files created:
  - data/mordor_attck_enriched.csv
  - attck/attack_mapping.json
  - attck/eventid_mapping.json
